In [3]:
import pandas as pd
from pathlib import Path

def build_ids(data_csv, manifest_csv, out_path):
    df = pd.read_csv(data_csv)
    manifest = pd.read_csv(manifest_csv)

    # merge on pair_id
    df = df.merge(
        manifest[["pair_id", "tcra_len", "tcrb_len"]],
        on="pair_id",
        how="left"
    )

    # -------------------------
    # PEPTIDE ID (simple)
    # -------------------------
    df["peptide_id"] = df["Peptide"].astype(str)

    # -------------------------
    # TCR ID (chain-aware)
    # -------------------------
    def make_tcr_id(row):
        seq = str(row["TCR_full"])

        # handle missing chains using manifest lengths
        has_alpha = row["tcra_len"] > 0
        has_beta = row["tcrb_len"] > 0

        if has_alpha and has_beta:
            return f"AB::{seq}"
        elif has_beta:
            return f"B::{seq}"
        elif has_alpha:
            return f"A::{seq}"
        else:
            return "UNK"

    df["tcr_id"] = df.apply(make_tcr_id, axis=1)

    # save
    df.to_csv(out_path, index=False)
    print(f"Saved: {out_path}")


# -------------------------
# RUN FOR ALL SPLITS
# -------------------------

build_ids(
    "/home/natasha/multimodal_model/data/train/train_df_clean.csv",
    "/home/natasha/multimodal_model/manifests/train_manifest.csv",
    "/home/natasha/multimodal_model/data/train/train_with_ids.csv"
)

build_ids(
    "/home/natasha/multimodal_model/data/val/val_df_clean_pos_neg.csv",
    "/home/natasha/multimodal_model/manifests/val_manifest.csv",
    "/home/natasha/multimodal_model/data/val/val_with_ids.csv"
)

build_ids(
    "/home/natasha/multimodal_model/data/test/test_df_clean_pos_neg.csv",
    "/home/natasha/multimodal_model/manifests/test_manifest.csv",
    "/home/natasha/multimodal_model/data/test/test_with_ids.csv"
)

Saved: /home/natasha/multimodal_model/data/train/train_with_ids.csv
Saved: /home/natasha/multimodal_model/data/val/val_with_ids.csv
Saved: /home/natasha/multimodal_model/data/test/test_with_ids.csv


In [5]:
df = pd.read_csv("/home/natasha/multimodal_model/data/train/train_with_ids.csv")

print(df["tcr_id"].nunique())
print(df["peptide_id"].nunique())

# check duplicates per peptide
print(df.groupby("peptide_id")["tcr_id"].nunique().describe())

29129
1084
count     1084.000000
mean        28.475092
std        378.574290
min          1.000000
25%          1.000000
50%          2.000000
75%          4.000000
max      11713.000000
Name: tcr_id, dtype: float64
